In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content="Hellow World",
    metadata={"source": "https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hellow World')

### Text data

In [5]:
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python.txt", encoding="utf-8")

/tmp/ipykernel_36831/2234444981.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [6]:
document = loader.load()

In [7]:
document

[Document(metadata={'source': 'data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

### Pdf Load

In [8]:
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research2.pdf")

# document = pdf_loader.load()

# document

In [9]:
# from langchain_community.document_loaders.pdf import PyMuPDFLoader # used for complex (contain images)

# pdf_loader = PyMuPDFLoader("data/research2.pdf")

# document = pdf_loader.load()

# document

# Ingestion Pipeline

In [10]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

### Documents

In [11]:
def load_all_pdfs():
    folder_path = "data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)

            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs
            

In [12]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [13]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### Chunks

In [14]:
# !pip install langchain_text_splitters

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [16]:
chunks = split_doc(all_pdf_documents)

In [17]:
len(chunks)

321

### Embeddings

In [18]:
from sentence_transformers import SentenceTransformer

In [19]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("model loading .....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions: ", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape", embeddings.shape)
        return embeddings

In [20]:
embedding_manager = EmbeddingManager()

model loading ..... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions:  384


/tmp/ipykernel_36831/2881715840.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions: ", self.model.get_sentence_embedding_dimension())


### Vector Store

In [21]:
import chromadb
import uuid

In [22]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.collection = None
        self.client = None

        self._initialize_store()


    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description":"vector store collection for pdf embeddings in RAG"}
        )
        
        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    
    
    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents doesnot match num of embeddings")

        # store => ids. embedding, documents, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_lenght"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            
            embeddings_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        print("total documents added in vector store:", len(documents_content))
        print("docs in collection:", self.collection.count())

In [23]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [24]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embeddings)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embeddings shape (321, 384)
total documents added in vector store: 321
docs in collection: 321


# Retrival Pipeline

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query -> embeddings
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # sementics search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine_similarity
        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadata = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadata, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "metadata": metadata,
                        "document": document,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i+1
                    })
            print(f"retrieved {len(retrieved_docs)} documents")
        else:
            print("no documnents found")
            
        return retrieved_docs

In [27]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [28]:
rag_retriever.retrieve("what is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape (1, 384)
retrieved 5 documents


[{'id': 'doc_338e1bcb-f150-4bc2-b4af-c11db66199bf',
  'metadata': {'trapped': '/False',
   'keywords': '',
   'page': 0,
   'total_pages': 21,
   'creator': 'LaTeX with hyperref',
   'content_lenght': 288,
   'doc_index': 88,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'source': 'data/research2.pdf',
   'title': '',
   'producer': 'pdfTeX-1.40.25',
   'author': '',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'moddate': '2024-03-28T00:54:45+00:00',
   'subject': '',
   'page_label': '1'},
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'distance': 0.46291494369506836,
  'similarity_score': 0.5370850563049316,
  'rank': 1},
 {'id': 'doc_d3bf6dab-6f10-

# Integrate with LLM

In [ ]:
API_KEY_GEMINI="put api"

In [31]:
# !pip install -qU  langchain-google-genai

In [34]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=API_KEY_GEMINI,
    temperature=0.1,  # creativity --> now we want valid(fact based)
    max_tokens=1024
)


### Generation of Retrieval-Augmented Output

In [39]:
def generate_output(query, rag_retriever, llm, top_k=3):
    results = rag_retriever.retrieve(query, top_k)

    context = "\n".join(doc["document"] for doc in results) if results else ""

    if not context:
        print("We found no relevant context for the given query")

    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt)
    return response.content[0]["text"]

In [43]:
answer = generate_output("what is encoder-decoder", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape (1, 384)
retrieved 3 documents


In [44]:
print(answer)

Based on the provided context, the encoder-decoder consists of the following stacks:

*   **Encoder:** 
    *   Composed of a stack of $N = 6$ identical layers.
    *   Each layer contains two sub-layers: the first is a multi-head self-attention mechanism, and the second is a simple, position- (the text cuts off here).
    *   It produces outputs of dimension $d_{model} = 512$.

*   **Decoder:** 
    *   Composed of a stack of $N = 6$ identical layers.
    *   It contains the same two sub-layers as the encoder, but inserts a third sub-layer
